# Data Science Historical Catastrophe
## Data Cleaning, Pipelines, and Classification

Your Name

---

## Getting Started

- Replace **Your Name** above with your full name
- Automatic 0 if you include your student ID or any other identifying number
- Rename the file to `Your_Name.ipynb` before submitting
- When finished, share your Colab link with **Anyone with the link can view** privileges
- Paste the shared link into the Canvas submission

---

## How to Use This Notebook

This notebook is designed to be accessible to all users, including those navigating with a **screen reader**.

### Screen Reader Navigation Guide

Every code section in this notebook gives you a choice. Before each code cell you will find a navigation landmark with two options:

- **Skip this code cell** — Jumps past the code directly to the output explanation. Use this if you want to focus on understanding results without reading raw code.
- **Go back and read the code cell** — Located after each explanation, this returns you to the top of the code section if you want to review it.

Each code cell is preceded by a plain-English description of what the code does, and followed by a description of the expected output.

**Tips for screen reader users:**
- Press **H** to jump between major section headings
- Press **K** or **Tab** to move between links, including the skip and return navigation links
- Press **D** to jump between landmark regions
- The skip and return links are inside labeled navigation landmarks — your screen reader will announce the landmark name so you always know where you are

---

## Learning Goals

By the end of this assignment you will be able to:

1. Generate a realistic synthetic dataset with intentional data quality problems
2. Conduct exploratory data analysis to identify distributions, correlations, and anomalies
3. Apply systematic data cleaning: remove constants, quasi-constants, duplicates, and handle missing values
4. Engineer derived features and encode categorical variables for modeling
5. Build and apply scikit-learn Pipelines to automate reproducible preprocessing workflows
6. Select features using multiple methods and justify your choices
7. Train and evaluate a logistic regression model using precision, recall, and bias-variance analysis
8. Interpret model results through the lens of a historical catastrophe narrative

---

## Deliverables

| Part | Title | Your Role |
| :--- | :--- | :--- |
| **Part 1** | The Data | Run all cells and observe what is being built |
| **Part 2** | The Story | Choose a historical event and map the data variables to it |
| **Part 3** | Exploratory Data Analysis | Explore, visualize, and describe the data using your story names |
| **Part 4** | Data Preparation | Clean the dataset — handle constants, duplicates, nulls, scaling, and outliers |
| **Part 5** | Feature Engineering | Create derived variables and encode categoricals |
| **Part 6** | Pipelines | Wrap your preprocessing workflow into a reproducible sklearn Pipeline |
| **Part 7** | Feature Selection | Use multiple selection methods to identify the best predictors |
| **Part 8** | Modeling and Evaluation | Build a logistic regression, evaluate it, and explain the results in writing |

---

## Instructions

- Follow each part in order. Do not skip ahead.
- On code cells with placeholder comments (e.g., `# YOUR CODE HERE`): write your solution directly in the cell.
- On commented-out code cells: uncomment and run them. Adapt column names to match your story mapping where needed.
- For written explanation cells: type directly into the Markdown cell. Write in complete sentences.
- After each code cell, read the explanation cell to confirm your output matches expectations.

---

# Part 1: The Data

In this part you will generate a synthetic dataset with 1,000 rows and approximately 38 columns. The data is deliberately messy — it contains constants, quasi-constants, duplicates, missing values, multicollinear features, outliers, and scaling problems. Your job in Parts 3 through 5 is to find and fix them.

**Run all cells in Part 1 in order without skipping.** They build on each other and must all complete successfully before you move to Part 2.

## Seed the Project

A unique random seed is generated from the current time so your dataset differs from everyone else's. The seed is stored as `user_seed` and reused throughout the notebook to keep results reproducible within your own session.

<a name="read-code-1"></a>

**Cell 1 of 21 — Generate a Unique Random Seed**

This cell creates a unique integer seed by combining the current system time (in nanoseconds) with a small random component, then applies it as the global numpy random seed. The printed number is your seed — it will differ from every other student's.

<nav aria-label="Code cell 1 navigation">
<a href="#skip-code-1" aria-label="Skip code cell 1 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import time
import numpy as np
import random

def generate_user_seed():
    nanoseconds = time.time_ns()
    random_component = random.randint(0, 1000)
    seed = nanoseconds ^ random_component
    seed = seed % (2**32)
    return seed

user_seed = generate_user_seed()
print(user_seed)
random_state = np.random.seed(user_seed)

<a name="skip-code-1"></a>

**Expected output:** A large integer, for example `2847391025`. This number is your session seed. Keep this notebook open so the seed remains in memory throughout all parts.

<nav aria-label="Return navigation for code cell 1">
<a href="#read-code-1" aria-label="Go back and read code cell 1">&#8617; Go back and read the code cell</a>
</nav>

## Install Faker

Faker is a Python library that generates realistic fake names, dates, and zip codes. It is used to create the demographic columns in the dataset.

<a name="read-code-2"></a>

**Cell 2 of 21 — Install the Faker Library**

This cell installs the Faker package using pip. The `-q` flag suppresses verbose output. This only needs to run once per Colab session.

<nav aria-label="Code cell 2 navigation">
<a href="#skip-code-2" aria-label="Skip code cell 2 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
pip install Faker -q

<a name="skip-code-2"></a>

**Expected output:** Minimal or no output if Faker is already installed. If you see an error, try restarting the Colab runtime and running again.

<nav aria-label="Return navigation for code cell 2">
<a href="#read-code-2" aria-label="Go back and read code cell 2">&#8617; Go back and read the code cell</a>
</nav>

## Define Location List

The dataset includes a categorical location column. This list provides the pool of location names the generator will sample from. **You will rename this list to fit your story in Part 2.**

<a name="read-code-3"></a>

**Cell 3 of 21 — Define the Location Pool**

This cell defines a Python list of 50 fictional planet and location names. When you map this to your historical story in Part 2, you will rename this list to something that fits your scenario (for example, `london_boroughs` or `roman_districts`).

<nav aria-label="Code cell 3 navigation">
<a href="#skip-code-3" aria-label="Skip code cell 3 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Rename this list to suit your story in Part 2
labels = [
    "Aster", "Bren", "Calo", "Drex", "Evin",
    "Faro", "Galen", "Hux", "Iven", "Jaro",
    "Kelis", "Lorn", "Mira", "Nexo", "Orin",
    "Pavo", "Quen", "Rilo", "Soren", "Tavi",
    "Ulan", "Vero", "Wex", "Yari", "Zeno",
    "Brin", "Cavi", "Daro", "Elix", "Fenn",
    "Garo", "Halen", "Irix", "Juno", "Kiro",
    "Lavi", "Moro", "Nari", "Olex", "Piri",
    "Qiro", "Riven", "Salo", "Toren", "Urix",
    "Vani", "Waro", "Xeno", "Yorin", "Zavi"
]

<a name="skip-code-3"></a>

**Expected output:** No output. The list is now in memory and available to the next cell.

<nav aria-label="Return navigation for code cell 3">
<a href="#read-code-3" aria-label="Go back and read code cell 3">&#8617; Go back and read the code cell</a>
</nav>

## Create Demographic Data

This cell builds the foundational 1,000-row DataFrame with demographic columns: two categorical fields, two name fields, a zip code, a date of birth, and a location.

<a name="read-code-4"></a>

**Cell 4 of 21 — Generate 1,000 Demographic Records**

This cell uses Faker to generate realistic-looking names and dates, and numpy to randomly assign biology type (`categorical_1`) and species (`categorical_2`). The result is a 1,000-row DataFrame called `demographics` with 7 columns. Note that `name_1` is female if `categorical_1` is Cytophore and male if Kymete — this is an intentional pattern you may want to address as potential bias in Part 8.

<nav aria-label="Code cell 4 navigation">
<a href="#skip-code-4" aria-label="Skip code cell 4 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import numpy as np
import pandas as pd
from faker import Faker
fake = Faker()

n = 1000
output = []
for x in range(n):
    biology = np.random.choice(['Cytophore', 'Kymete'], p=[0.5, 0.5])
    output.append({
        'categorical_1': biology,
        'categorical_2': np.random.choice(['Xylosian', 'Veridian', 'CKaeltharr']),
        'name_1': fake.first_name_female() if biology == 'Cytophore' else fake.first_name_male(),
        'name_2': fake.last_name(),
        'code': fake.zipcode(),
        'date': fake.date_of_birth(),
        'labels': np.random.choice(labels)
    })

demographics = pd.DataFrame(output)
print(demographics.shape)
demographics.head()

<a name="skip-code-4"></a>

**Expected output:** `(1000, 7)` followed by a preview of the first 5 rows. You should see columns for categorical_1, categorical_2, name_1, name_2, code, date, and location.

<nav aria-label="Return navigation for code cell 4">
<a href="#read-code-4" aria-label="Go back and read code cell 4">&#8617; Go back and read the code cell</a>
</nav>

## Feature Generation Functions

The next two cells define the classification target and a helper function that generates correlated feature data. Run them — you do not need to modify them.

<a name="read-code-5"></a>

**Cell 5 of 21 — Define the Feature Generation Helper**

This cell defines `generate_feature()`, a function that creates a normally distributed feature correlated with a binary class column. It uses inverse logit (logit transformation) to ensure the generated values are statistically linked to the target class.

<nav aria-label="Code cell 5 navigation">
<a href="#skip-code-5" aria-label="Skip code cell 5 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import numpy as np
import pandas as pd

def generate_feature(df, class_col, coeff, intercept):
    probs = np.random.rand(len(df))
    probs = np.where(df[class_col] == 1, probs * 0.8 + 0.2, probs * 0.8)
    logits = np.log(probs / (1 - probs))
    feature_values = (logits - intercept) / coeff
    return pd.Series(feature_values)

<a name="skip-code-5"></a>

**Expected output:** No output. The function is now available in memory.

<nav aria-label="Return navigation for code cell 5">
<a href="#read-code-5" aria-label="Go back and read code cell 5">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-6"></a>

**Cell 6 of 21 — Generate Classification Target and Informative Features**

This cell uses scikit-learn's `make_classification` to create two informative numerical features (`informative_1` and `informative_2`) and a binary target (`class`). A logistic regression is immediately fit to those features to create a continuous `target` variable (the log-odds score). A third feature correlated with class is also generated. All are merged with the demographics DataFrame.

<nav aria-label="Code cell 6 navigation">
<a href="#skip-code-6" aria-label="Skip code cell 6 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

def make_linear_y(row):
    model = LogisticRegression()
    model.fit(X, y)
    coefficients = model.coef_
    intercept = model.intercept_
    f_of_x = intercept + coefficients[0][0]*row['informative_1'] + coefficients[0][1]*row['informative_2']
    return f_of_x[0]

X, y = make_classification(n_samples=n, n_features=2, n_informative=2,
                            n_redundant=0, n_repeated=0, random_state=42)
df = pd.DataFrame(X, columns=['informative_1', 'informative_2'])
df = pd.concat([demographics, df], axis=1).reset_index(drop=True)
df['target'] = df.apply(make_linear_y, axis=1)
df['class'] = y
df['corr_feature_class'] = generate_feature(df, 'class', 0.5, -1)
df.head()

<a name="skip-code-6"></a>

**Expected output:** A preview of the first 5 rows showing the demographic columns plus `informative_1`, `informative_2`, `target`, `class`, and `corr_feature_class`. The `class` column should contain only 0s and 1s.

<nav aria-label="Return navigation for code cell 6">
<a href="#read-code-6" aria-label="Go back and read code cell 6">&#8617; Go back and read the code cell</a>
</nav>

## Automation Functions

The following cell defines 11 helper functions used to generate the messy features. These are provided for you — run the cell and continue.

<a name="read-code-7"></a>

**Cell 7 of 21 — Define All Data Generation Helper Functions**

This cell defines 11 functions: `gen_null` (introduces missing values), `gen_quasi_constants` (near-constant columns), `gen_normal_data`, `gen_uniform_data`, `gen_multivariate_normal_data` (correlated feature pairs), `gen_correlated_normal_series`, `gen_correlated_uniform_series`, `gen_outliers`, `gen_standard_scaling`, `gen_minmax_scaling`, and `random_choice_data`. None of them run yet — they are stored in memory for the cells that follow.

<nav aria-label="Code cell 7 navigation">
<a href="#skip-code-7" aria-label="Skip code cell 7 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import minimize

def gen_null(series, perc):
    var = series.copy()
    num_nulls = int(len(var) * (perc / 100))
    indices_to_replace = np.random.choice(len(var), num_nulls, replace=False)
    for idx in indices_to_replace:
        var[idx] = np.nan
    return var

def gen_quasi_constants(primary_label, variation_percentage=.2, size=len(df)):
    series = pd.Series(np.full(size, primary_label))
    num_variations = int(size * (variation_percentage / 100))
    variation_indices = np.random.choice(series.index, num_variations, replace=False)
    primary_label = primary_label + '_0'
    variation1 = primary_label + '_1'
    variation2 = primary_label + '_2'
    labels = pd.Series([primary_label] * len(series), index=series.index)
    labels.loc[variation_indices] = np.random.choice([variation1, variation2], size=num_variations)
    return labels

def gen_normal_data(mu=0, std=1, size=len(df)):
    return np.random.normal(mu, std, size)

def gen_uniform_data(size=len(df)):
    return np.random.uniform(size=size)

def gen_multivariate_normal_data(mean=[0,0], cov=[[1,0],[0,1]], size=len(df)):
    ds1, ds2 = np.random.multivariate_normal(mean, cov, size, tol=1e-6).T
    return ds1, ds2

def gen_correlated_normal_series(original_series, target_correlation, size=len(df)):
    return (np.mean(original_series)
            + target_correlation * (original_series - np.mean(original_series))
            + np.random.normal(0, np.sqrt(1 - target_correlation**2) * np.std(original_series),
                               len(original_series)))

def gen_correlated_uniform_series(original_series, correlation_coefficient=0, size=len(df)):
    z_scores = (original_series - np.mean(original_series)) / np.std(original_series)
    correlation_coefficient = .7
    return norm.cdf(correlation_coefficient * norm.ppf(np.random.uniform(size=size))
                   + np.sqrt(1 - correlation_coefficient**2) * z_scores)

def gen_outliers(mean=0, std_dev=1, size=len(df), outlier_percentage=0.1, outlier_magnitude=3):
    data = np.random.normal(mean, std_dev, size)
    num_outliers = int(size * outlier_percentage)
    outlier_indices = np.random.choice(size, num_outliers, replace=False)
    for index in outlier_indices:
        data[index] += outlier_magnitude if np.random.rand() < 0.5 else -outlier_magnitude
    return data

def gen_standard_scaling(mean=50, std_dev=10, size=len(df), scale_factor=1000):
    return np.random.normal(mean, std_dev, size) * scale_factor

def gen_minmax_scaling(mean=50, std_dev=10, size=len(df), range_factor=10):
    original_data = np.random.normal(mean, std_dev, size)
    min_val = np.min(original_data)
    max_val = np.max(original_data)
    return (original_data - min_val) * range_factor + min_val

def random_choice_data(choices, size):
    return np.random.choice(choices, size=size)

print('All generation functions loaded.')

<a name="skip-code-7"></a>

**Expected output:** `All generation functions loaded.` All 11 functions are now available in memory.

<nav aria-label="Return navigation for code cell 7">
<a href="#read-code-7" aria-label="Go back and read code cell 7">&#8617; Go back and read the code cell</a>
</nav>

## Build the Messy Dataset

The next series of cells adds all the intentionally flawed columns to the DataFrame. Run each in order.

<a name="read-code-8"></a>

**Cell 8 of 21 — Add Categorical Noise Features**

This cell adds five categorical columns with no meaningful correlation to the target: two binary random choices, a day-of-week column, and two columns with randomly determined numbers of labels (one with few labels, one with many). These are 'noise' features your feature selection work in Part 7 should identify and discard.

<nav aria-label="Code cell 8 navigation">
<a href="#skip-code-8" aria-label="Skip code cell 8 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
df['random choice 2'] = random_choice_data(['Rand Choice 1', 'Rand Choice 2'], size=len(df))
df['random choice 4'] = random_choice_data(['North', 'South', 'East', 'West'], size=len(df))
df['random choice 7'] = random_choice_data(
    ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'], size=len(df))
num_labels = np.random.randint(3, 5)
df[f'random label num {num_labels}'] = random_choice_data(
    [f'label num lo {i}' for i in range(1, num_labels + 1)], size=len(df))
num_labels = np.random.randint(10, 15)
df[f'random label num {num_labels}'] = random_choice_data(
    [f'label num hi {i}' for i in range(1, num_labels + 1)], size=len(df))
print('Noise categoricals added. Shape:', df.shape)

<a name="skip-code-8"></a>

**Expected output:** `Noise categoricals added. Shape: (1000, N)` where N has increased. Five new columns have been added.

<nav aria-label="Return navigation for code cell 8">
<a href="#read-code-8" aria-label="Go back and read code cell 8">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-9"></a>

**Cell 9 of 21 — Add Categorical Features Correlated with the Target**

This cell uses `pd.qcut` to bin the continuous `target` variable into discrete categories. `pd qcut1` creates a binary Low/High split, `pd qcut2` creates four quartile labels, and `pd qcut3` creates six unequal bins. Because they are derived from `target`, these are correlated with `class` and should be identified as useful predictors in Part 7.

<nav aria-label="Code cell 9 navigation">
<a href="#skip-code-9" aria-label="Skip code cell 9 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
df['pd qcut1'] = pd.qcut(df['target'], 2, labels=['Low', 'High'])
df['pd qcut2'] = pd.qcut(df['target'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
quantiles = [0, 0.1, 0.2, 0.4, 0.6, 0.8, 1]
df['pd qcut3'] = pd.qcut(df['target'], quantiles, labels=['G1','G2','G3','G4','G5','G6'])
print('Correlated categoricals added. Shape:', df.shape)

<a name="skip-code-9"></a>

**Expected output:** Three new columns added. These bins are based on `target`, so they carry predictive signal for `class`.

<nav aria-label="Return navigation for code cell 9">
<a href="#read-code-9" aria-label="Go back and read code cell 9">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-10"></a>

**Cell 10 of 21 — Add Multicollinear Feature Pairs**

This cell generates two pairs of highly correlated numerical features using a multivariate normal distribution. `multicollinearity 1` and `multicollinearity 2` share a correlation of 0.7. `multicollinearity 3` and `multicollinearity 4` share a correlation of 0.9. These deliberately violate the assumption of feature independence and must be addressed in Part 4.

<nav aria-label="Code cell 10 navigation">
<a href="#skip-code-10" aria-label="Skip code cell 10 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
df['multicollinearity 1'], df['multicollinearity 2'] = gen_multivariate_normal_data(
    mean=[0, 0], cov=[[1, .7], [.7, 1]], size=len(df))
df['multicollinearity 3'], df['multicollinearity 4'] = gen_multivariate_normal_data(
    mean=[0, 0], cov=[[1, .9], [.9, 1]], size=len(df))
print('Multicollinear features added. Shape:', df.shape)

<a name="skip-code-10"></a>

**Expected output:** Four new columns. You will verify their correlations with the heatmap in Part 3.

<nav aria-label="Return navigation for code cell 10">
<a href="#read-code-10" aria-label="Go back and read code cell 10">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-11"></a>

**Cell 11 of 21 — Add Features Correlated with the Target**

This cell adds four features with known correlations to `target`: two normally distributed (target correlations of 0.5 and 0.7) and two uniformly distributed (using rank correlation). These should surface in your feature selection analysis in Part 7.

<nav aria-label="Code cell 11 navigation">
<a href="#skip-code-11" aria-label="Skip code cell 11 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
df['correlated w target 1'] = gen_correlated_normal_series(df['target'], target_correlation=.5)
df['correlated w target 2'] = gen_correlated_normal_series(df['target'], target_correlation=.7)
df['uniform corr 1'] = gen_correlated_uniform_series(df['target'])
df['uniform corr 2'] = gen_correlated_uniform_series(df['target'])
df.info()

<a name="skip-code-11"></a>

**Expected output:** `df.info()` output showing all columns so far. Four new correlated features should appear.

<nav aria-label="Return navigation for code cell 11">
<a href="#read-code-11" aria-label="Go back and read code cell 11">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-12"></a>

**Cell 12 of 21 — Add Duplicate Features**

This cell creates exact copies of `informative_1` and `informative_2`. Duplicate features are a common data quality problem — they inflate the apparent feature count without adding information and can destabilize coefficient estimates. You will detect and remove them in Part 4.

<nav aria-label="Code cell 12 navigation">
<a href="#skip-code-12" aria-label="Skip code cell 12 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
df['duplicate_1'] = df['informative_1']
df['duplicate_2'] = df['informative_2']
print('Duplicate features added. Shape:', df.shape)

<a name="skip-code-12"></a>

**Expected output:** Two new columns that are identical to `informative_1` and `informative_2`. Your cleaning step in Part 4 will detect and remove these.

<nav aria-label="Return navigation for code cell 12">
<a href="#read-code-12" aria-label="Go back and read code cell 12">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-13"></a>

**Cell 13 of 21 — Add Outlier Features**

This cell generates two numerical features with injected extreme values. `outliers 1` has 10% outliers of magnitude 3 standard deviations. `outliers 2` has 20% outliers of magnitude 2 standard deviations with a shifted mean. You will visualize and treat these in Parts 3 and 4.

<nav aria-label="Code cell 13 navigation">
<a href="#skip-code-13" aria-label="Skip code cell 13 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
df['outliers 1'] = gen_outliers(mean=0, std_dev=1, size=len(df),
                                outlier_percentage=0.1, outlier_magnitude=3)
df['outliers 2'] = gen_outliers(mean=3, std_dev=2, size=len(df),
                                outlier_percentage=0.2, outlier_magnitude=2)
print('Outlier features added. Shape:', df.shape)

<a name="skip-code-13"></a>

**Expected output:** Two new columns with visible extreme values. You will quantify the outliers using the IQR method in Part 3.

<nav aria-label="Return navigation for code cell 13">
<a href="#read-code-13" aria-label="Go back and read code cell 13">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-14"></a>

**Cell 14 of 21 — Add Features Needing Scaling**

This cell generates two features at very different scales from the rest of the dataset. `standard scaling` has values in the thousands (requiring StandardScaler). `min max scaling` has an expanded range (requiring MinMaxScaler). In Part 4 you will apply the correct scaler to each.

<nav aria-label="Code cell 14 navigation">
<a href="#skip-code-14" aria-label="Skip code cell 14 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
df['standard scaling'] = gen_standard_scaling()
df['min max scaling'] = gen_minmax_scaling()
print('Scaling features added. Shape:', df.shape)

<a name="skip-code-14"></a>

**Expected output:** Two new columns. Notice that `standard scaling` values are roughly 1,000 times larger than the other numerical features.

<nav aria-label="Return navigation for code cell 14">
<a href="#read-code-14" aria-label="Go back and read code cell 14">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-15"></a>

**Cell 15 of 21 — Inject Missing Values**

This cell randomly sets a percentage of values to null in most columns. The percentage is randomly chosen per column from the set [0, 5, 10, 20, 30, 50]. Key columns (`class`, `informative_1`, `informative_2`, `target`, and the duplicates) are protected. You will diagnose and impute these nulls in Part 4.

<nav aria-label="Code cell 15 navigation">
<a href="#skip-code-15" aria-label="Skip code cell 15 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
for col in df.drop(['class','informative_1','informative_2','target','duplicate_1','duplicate_2'],
                   axis=1).columns:
    df[col] = gen_null(df[col], np.random.choice([0, 5, 10, 20, 30, 50], size=1).item())
print('Nulls injected. Missing value counts:')
print(df.isnull().sum()[df.isnull().sum() > 0])

<a name="skip-code-15"></a>

**Expected output:** A list of columns with their null counts. Some columns may have no nulls (0% was a possible choice); others may be missing up to 50% of values.

<nav aria-label="Return navigation for code cell 15">
<a href="#read-code-15" aria-label="Go back and read code cell 15">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-16"></a>

**Cell 16 of 21 — Add Constant and Quasi-Constant Features**

This cell adds four zero-variance or near-zero-variance columns. `constant_1` and `constant_2` have the same value in every row (zero variance). `semi_constant_1` and `semi_constant_2` have one dominant label in 99% of rows. These carry no predictive signal and will be removed in Part 4.

<nav aria-label="Code cell 16 navigation">
<a href="#skip-code-16" aria-label="Skip code cell 16 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
df['constant_1'] = 'constant_value'
df['constant_2'] = 'constant_value'
df['semi_constant_1'] = gen_quasi_constants('q_const', variation_percentage=1)
df['semi_constant_2'] = gen_quasi_constants('q_const', variation_percentage=1)
print('Constant and quasi-constant features added. Shape:', df.shape)

<a name="skip-code-16"></a>

**Expected output:** Four new columns. `constant_1` and `constant_2` will show `nunique() == 1`. `semi_constant_1` and `semi_constant_2` will show one label with ~99% frequency.

<nav aria-label="Return navigation for code cell 16">
<a href="#read-code-16" aria-label="Go back and read code cell 16">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-17"></a>

**Cell 17 of 21 — Verify Column Count**

This cell runs `df.info()` to confirm the dataset shape and column types before final assembly.

<nav aria-label="Code cell 17 navigation">
<a href="#skip-code-17" aria-label="Skip code cell 17 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
print(df.info())

<a name="skip-code-17"></a>

**Expected output:** A summary of approximately 38 columns and 1,000 rows. You should see a mix of float64 and object dtypes.

<nav aria-label="Return navigation for code cell 17">
<a href="#read-code-17" aria-label="Go back and read code cell 17">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-18"></a>

**Cell 18 of 21 — Add Duplicate Rows and Shuffle Columns**

This cell appends the first 10 rows as duplicate records, then shuffles the non-demographic column order randomly. This ensures every student's column arrangement is different, so your variable mapping in Part 2 is unique to your session.

<nav aria-label="Code cell 18 navigation">
<a href="#skip-code-18" aria-label="Skip code cell 18 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
dupes = df.loc[0:9]
df = pd.concat([df, dupes], axis=0)
demographic_columns = demographics.columns
remaining_columns = [col for col in df.columns if col not in demographic_columns]
np.random.shuffle(remaining_columns)
df = df[list(demographic_columns) + list(remaining_columns)]
class_var = 'class'
df = df[df.drop('class', axis=1).columns.tolist() + [class_var]]
print(df.shape)
print(df.info())
df.head()

<a name="skip-code-18"></a>

**Expected output:** Shape should now show 1,010 rows (1,000 original + 10 duplicates). Column order will differ from classmates — that is intentional.

<nav aria-label="Return navigation for code cell 18">
<a href="#read-code-18" aria-label="Go back and read code cell 18">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-19"></a>

**Cell 19 of 21 — Save Part 1 Dataset**

This cell saves the raw messy dataset to a CSV file called `data science fiction iv pt 1.csv`. Part 3 (EDA) will load from this file. Do not skip this step.

<nav aria-label="Code cell 19 navigation">
<a href="#skip-code-19" aria-label="Skip code cell 19 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
df.to_csv('data science fiction iv pt 1.csv', index=False)
print('Saved.')

<a name="skip-code-19"></a>

**Expected output:** `Saved.` The CSV file is now in your Colab session storage. You can verify it exists by running `!ls` in a new cell.

<nav aria-label="Return navigation for code cell 19">
<a href="#read-code-19" aria-label="Go back and read code cell 19">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 2: The Story

Your dataset is a blank canvas. In this part you will choose a historical catastrophe and give every column a name and meaning that fits your narrative.

## Choose Your Historical Catastrophe

**Step 1 — Choose a scenario.** Pick a historical catastrophe with a clear binary outcome. You may use one of the examples below or propose your own.

| Scenario | Class = 0 (Normal) | Class = 1 (Catastrophic) |
| :--- | :--- | :--- |
| The Great Smog of London (1952) | Baseline mortality | High-mortality event |
| The Bronze Age Collapse (c. 1177 BC) | Settlement survival | Settlement collapse |
| The Great Fire of Rome (64 AD) | Surviving structure | Total incineration |
| The Black Death (1347–1353) | Routine illness | Confirmed plague case |
| The Irish Famine (1845–1852) | Seasonal crop variation | Famine onset |

**Step 2 — Generate your story using AI.** Copy your `df.info()` output from Part 1 into an AI assistant (Claude or ChatGPT) with this prompt:

> *I have a dataset I am using for a historical catastrophe story about [your chosen event]. Here is my column list: [paste df.info() output]. Map each column to a story variable name that fits the scenario and explain why it fits. Define class = 0 as the null hypothesis (normal state) and class = 1 as the catastrophic event.*

**Step 3 — Paste the AI's response below** and complete the variable mapping table.

---

## Paste Your Story Here

*(Replace this line with your historical narrative — at least two paragraphs.)*

---

## Things to Consider

### Defining Your Hypotheses

| Hypothesis | Statistical Term | Story Meaning |
| :--- | :--- | :--- |
| **H₀** | Class = 0 | The normal state — nothing extraordinary is happening |
| **H₁** | Class = 1 | The catastrophic event — the disaster is underway |

### The Cost of Error

| Error Type | Statistical Term | Story Consequence |
| :--- | :--- | :--- |
| **Type I (α)** | False Positive | Raised the alarm when there was no crisis — wasted resources or caused panic |
| **Type II (β)** | False Negative | Missed the real catastrophe — disaster spread unchecked |

Think about which error is more costly for your specific scenario. You will revisit this in Part 8 when evaluating precision and recall.

---

## Variable Mapping Table

Rename every column from your `df.info()` output to fit your story. **Use these renamed variables consistently in Parts 3 through 8.**

| Original Variable Name | Story Variable Name | Why It Fits |
| :--- | :--- | :--- |
| date | | |
| location | | |
| target | | |
| class | | |
| informative_1 | | |
| informative_2 | | |
| *add all remaining columns* | | |

*(Add as many rows as you have columns.)*

---
# Part 3: Exploratory Data Analysis

Exploratory Data Analysis (EDA) is the process of understanding your data before modeling. In this part you will examine variable types, distributions, correlations, and outliers. Use your story variable names throughout — not the original system names.

**Run the Load Data cell first before attempting any other cell in this part.**

<a name="read-code-20"></a>

**Cell 20 — Load the Part 1 Dataset**

This cell reads the CSV saved at the end of Part 1 into a fresh DataFrame. It prints the shape and column summary so you can confirm the data loaded correctly.

<nav aria-label="Code cell 20 navigation">
<a href="#skip-code-20" aria-label="Skip code cell 20 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import pandas as pd

df = pd.read_csv('data science fiction iv pt 1.csv')
print(df.shape)
print(df.info())
df.head()

<a name="skip-code-20"></a>

**Expected output:** Shape of (1010, ~38), a column listing, and a 5-row preview. If you see a FileNotFoundError, return to Part 1 and run Cell 19.

<nav aria-label="Return navigation for code cell 20">
<a href="#read-code-20" aria-label="Go back and read code cell 20">&#8617; Go back and read the code cell</a>
</nav>

## Variable Types

Knowing each column's data type is the foundation of all downstream decisions. The type determines which statistics are meaningful, which plots are appropriate, and which encoding or scaling strategy applies.

| Type | What it means | Key operations |
| :--- | :--- | :--- |
| **Numerical (float/int)** | Measurable quantities | Mean, std, correlation, scaling |
| **Object (string)** | Text or unordered labels | Mode, frequency, encoding |
| **Category** | Ordered or unordered discrete groups | Frequency, ordinal encoding |

<a name="read-code-21"></a>

**Cell 21 — Identify Variable Types**

This cell uses `select_dtypes()` to partition columns into numerical, object (string), and category types. The results are stored in four variables used throughout Parts 3 and 4.

<nav aria-label="Code cell 21 navigation">
<a href="#skip-code-21" aria-label="Skip code cell 21 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
df_numerical = df.select_dtypes(include='number').columns
df_object = df.select_dtypes(include=['object']).columns
df_discrete = df.select_dtypes(include=['category']).columns
df_categorical_features = df.select_dtypes(include=['category', 'object']).columns
print('Numerical:', list(df_numerical))
print('Object:', list(df_object))
print('Category:', list(df_discrete))

<a name="skip-code-21"></a>

**Expected output:** Three lists showing which columns fall into each type. Notice that `pd qcut` columns appear as category dtype, while name, location, and random choice columns appear as object.

<nav aria-label="Return navigation for code cell 21">
<a href="#read-code-21" aria-label="Go back and read code cell 21">&#8617; Go back and read the code cell</a>
</nav>

### ✏️ Your Turn — Variable Type Visualizations

Uncomment and run the two cells below. Replace the column names with your story variable names. After running each, add a one-sentence comment in the code cell describing what you observe (for example: is the variable balanced? skewed? dominated by one label?)

<a name="read-code-22"></a>

**Cell 22 — Histogram of the Target Variable**

This commented cell will draw a histogram with a KDE overlay for your `target` column (use your story name). The shape reveals whether the continuous target is symmetric, skewed, or bimodal — which affects how well a linear model will fit.

<nav aria-label="Code cell 22 navigation">
<a href="#skip-code-22" aria-label="Skip code cell 22 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# plt.figure(figsize=(10, 6))
# sns.histplot(df['target'], kde=True, color='blue')  # Replace 'target' with your story name
# plt.title('Distribution of Target Variable')  # Update title
# plt.xlabel('Target Value')  # Update axis label
# plt.ylabel('Frequency')
# plt.show()

# Observation: (write your one-sentence comment here after running)

<a name="skip-code-22"></a>

**Expected output:** A bell-shaped or near-symmetric histogram with KDE curve. If it is heavily skewed, note that in your comment — this is relevant to outlier treatment in Part 4.

**Accessibility note:** Add a Markdown cell below this one describing the chart in plain language, for example: 'The histogram shows a roughly normal distribution centered near zero with a small right tail.'

<nav aria-label="Return navigation for code cell 22">
<a href="#read-code-22" aria-label="Go back and read code cell 22">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-23"></a>

**Cell 23 — Count Plot of a Categorical Variable**

This commented cell will draw a bar chart showing the frequency of each level in one of your categorical columns. Use it to check for class imbalance or dominant labels.

<nav aria-label="Code cell 23 navigation">
<a href="#skip-code-23" aria-label="Skip code cell 23 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# plt.figure(figsize=(10, 6))
# sns.countplot(x='categorical_1', data=df,  # Replace with your story name
#               order=df['categorical_1'].value_counts().index)
# plt.title('Frequency of Categorical_1 Levels')  # Update title
# plt.xticks(rotation=45)
# plt.show()

# Observation: (write your one-sentence comment here after running)

<a name="skip-code-23"></a>

**Expected output:** A bar chart with roughly equal bars for balanced categoricals or one dominant bar for near-constant columns. Near-constant columns should be removed in Part 4.

**Accessibility note:** Add a Markdown cell below describing what the chart shows about the balance of your chosen categorical variable.

<nav aria-label="Return navigation for code cell 23">
<a href="#read-code-23" aria-label="Go back and read code cell 23">&#8617; Go back and read the code cell</a>
</nav>

## Correlation Analysis

Correlation measures the linear relationship between two numerical variables. In EDA we use it for two purposes:

- **Feature-target correlation:** High |ρ| with `class` suggests a useful predictor.
- **Feature-feature correlation:** High |ρ| between features signals multicollinearity — a problem for modeling.

$$\rho_{X,Y} = \frac{\text{cov}(X,Y)}{\sigma_X \sigma_Y}$$

Values range from −1 (perfect negative) to +1 (perfect positive). Make note of any pairs with |ρ| > 0.7 — you will need to address those in Part 4.

<a name="read-code-24"></a>

**Cell 24 — Raw Correlation Matrix**

This commented cell displays the Pearson correlation matrix for all numerical columns. It is a starting point before the visual heatmap. Uncomment and run to see the raw values.

<nav aria-label="Code cell 24 navigation">
<a href="#skip-code-24" aria-label="Skip code cell 24 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Uncomment and run to view the raw correlation matrix
# df[df_numerical].corr().round(2)

<a name="skip-code-24"></a>

**Expected output:** A large matrix of correlation values between −1 and +1. Look for off-diagonal values with large magnitude — those are your multicollinearity candidates.

<nav aria-label="Return navigation for code cell 24">
<a href="#read-code-24" aria-label="Go back and read code cell 24">&#8617; Go back and read the code cell</a>
</nav>

### ✏️ Your Turn — Correlation Heatmap

<a name="read-code-25"></a>

**Cell 25 — Correlation Heatmap**

This commented cell draws a color-coded heatmap of the correlation matrix. Dark red cells indicate strong positive correlation; dark blue indicates strong negative. You should be able to visually identify the multicollinear pairs injected in Part 1.

<nav aria-label="Code cell 25 navigation">
<a href="#skip-code-25" aria-label="Skip code cell 25 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns

# sns.set(style='white')
# corr = df[df_numerical].corr().round(1)
# mask = np.zeros_like(corr, dtype=bool)
# mask[np.triu_indices_from(mask)] = True
# f = plt.figure(figsize=(12, 12))
# cmap = sns.diverging_palette(220, 10, as_cmap=True)
# sns.heatmap(corr, mask=mask, cmap=cmap, vmax=.3, center=0,
#             square=True, linewidths=.5, cbar_kws={'shrink': .5}, annot=True)
# plt.tight_layout()

# Observation: (note which feature pairs show high correlation)

<a name="skip-code-25"></a>

**Expected output:** A lower-triangle heatmap. You should see two clusters of high correlation: the multicollinearity 1/2 pair and the 3/4 pair.

**Accessibility note:** Add a Markdown cell below describing the key pattern: which pairs are highly correlated and which features appear independent.

<nav aria-label="Return navigation for code cell 25">
<a href="#read-code-25" aria-label="Go back and read code cell 25">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-26"></a>

**Cell 26 — Identify Highly Correlated Feature Pairs**

This commented cell converts the correlation matrix to a long format and filters for pairs with |ρ| > 0.7. The result is a ranked list of the most concerning feature pairs.

<nav aria-label="Code cell 26 navigation">
<a href="#skip-code-26" aria-label="Skip code cell 26 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# corr_matrix = df[df_numerical].corr()
# corr_df = corr_matrix.stack().reset_index()
# corr_df.columns = ['feature1', 'feature2', 'correlation']
# high_corr_df = corr_df[
#     (abs(corr_df['correlation']) > 0.7) & (corr_df['feature1'] != corr_df['feature2'])]
# high_corr_df = high_corr_df.sort_values(by='correlation', ascending=False, key=abs)
# print(high_corr_df)

# Note the highly correlated pairs here:
# Pair 1:
# Pair 2:
# Pair 3:

<a name="skip-code-26"></a>

**Expected output:** A table of feature pairs sorted by absolute correlation. Note the pairs you will need to address in Part 4 — keep only one from each correlated pair.

<nav aria-label="Return navigation for code cell 26">
<a href="#read-code-26" aria-label="Go back and read code cell 26">&#8617; Go back and read the code cell</a>
</nav>

### ✏️ Your Turn — VIF Check

<a name="read-code-27"></a>

**Cell 27 — Variance Inflation Factor (VIF)**

This commented cell calculates VIF for every numerical column. VIF measures how much a feature's variance is inflated by its correlation with other features. VIF > 5 is a warning; VIF > 10 indicates severe multicollinearity. Features above the threshold should be candidates for removal.

<nav aria-label="Code cell 27 navigation">
<a href="#skip-code-27" aria-label="Skip code cell 27 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# from statsmodels.stats.outliers_influence import variance_inflation_factor

# x_copy = df.drop('class', axis=1)._get_numeric_data()
# x_copy.fillna(x_copy.mean(), inplace=True)

# vif = pd.DataFrame()
# vif['Variable'] = x_copy.columns
# vif['VIF'] = [variance_inflation_factor(x_copy, i) for i in range(x_copy.shape[1])]
# print(vif.sort_values('VIF', ascending=False))

# Features exceeding VIF > 10: (list them here)

<a name="skip-code-27"></a>

**Expected output:** A DataFrame with VIF scores per feature. The multicollinear pairs from Part 1 should show the highest VIF values. Keep a note of these — you will remove them iteratively in the next cell.

<nav aria-label="Return navigation for code cell 27">
<a href="#read-code-27" aria-label="Go back and read code cell 27">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-28"></a>

**Cell 28 — Iterative VIF Removal**

This commented cell repeatedly removes the feature with the highest VIF until all remaining features fall below the threshold of 10. The list of removed features is stored in `removed` for reference.

<nav aria-label="Code cell 28 navigation">
<a href="#skip-code-28" aria-label="Skip code cell 28 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# from statsmodels.stats.outliers_influence import variance_inflation_factor

# removed = []
# x_copy1 = x_copy.copy()
# max_vif = thresh = 10
# while max_vif >= thresh:
#     my_list = [variance_inflation_factor(x_copy1, i) for i in range(x_copy1.shape[1])]
#     max_vif = max(my_list)
#     if max_vif > thresh:
#         max_index = my_list.index(max_vif)
#         removed.append(x_copy1.columns[max_index])
#         print(x_copy1.columns[max_index], variance_inflation_factor(x_copy1, max_index))
#         x_copy1.drop(x_copy1.columns[max_index], axis=1, inplace=True)
# vif = pd.DataFrame()
# vif['Variable'] = x_copy1.columns
# vif['VIF'] = [variance_inflation_factor(x_copy1, i) for i in range(x_copy1.shape[1])]
# print(vif)
# print('Removed:', removed)

<a name="skip-code-28"></a>

**Expected output:** A log of removed features followed by the final VIF table where all values are below 10. Reference the `removed` list when building your final feature set in Part 7.

<nav aria-label="Return navigation for code cell 28">
<a href="#read-code-28" aria-label="Go back and read code cell 28">&#8617; Go back and read the code cell</a>
</nav>

## Outliers

Outliers are values that fall far from the typical range. They can bias statistical estimates, distort scaling, and cause gradient instability during model training. The IQR method flags values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR as outliers.

$$IQR = Q_3 - Q_1$$

### ✏️ Your Turn — Outlier Exploration

<a name="read-code-29"></a>

**Cell 29 — Boxplot of an Outlier Column**

This commented cell draws a boxplot for one of your outlier-heavy columns. The dots beyond the whiskers are values flagged as outliers by the IQR rule. Replace the column name with your story variable name.

<nav aria-label="Code cell 29 navigation">
<a href="#skip-code-29" aria-label="Skip code cell 29 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# import matplotlib.pyplot as plt

# df.boxplot(column=['outliers 1'])  # Replace with your story name
# plt.title('Boxplot — Outlier Column')  # Update title
# plt.show()

# Observation: (describe what the boxplot reveals about the severity and direction of outliers)

<a name="skip-code-29"></a>

**Expected output:** A boxplot with visible dots beyond the upper and/or lower whiskers. The IQR box should be compact with outliers appearing as scattered points.

**Accessibility note:** Add a Markdown cell below describing the chart: where the box sits, how long the whiskers are, and approximately how many outlier points are visible.

<nav aria-label="Return navigation for code cell 29">
<a href="#read-code-29" aria-label="Go back and read code cell 29">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-30"></a>

**Cell 30 — Outlier Count by Column (IQR Method)**

This commented cell defines and runs a function that counts outliers in every numerical column using the IQR method. Run `df.describe()` first to check min/max values, then run the outlier counter.

<nav aria-label="Code cell 30 navigation">
<a href="#skip-code-30" aria-label="Skip code cell 30 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# df.describe()

# def count_outliers_iqr(df, column):
#     Q1 = df[column].quantile(0.25)
#     Q3 = df[column].quantile(0.75)
#     IQR = Q3 - Q1
#     lower_bound = Q1 - 1.5 * IQR
#     upper_bound = Q3 + 1.5 * IQR
#     outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
#     return len(outliers)

# for col in df[df_numerical].columns:
#     n = count_outliers_iqr(df, col)
#     if n > 0:
#         print(f'{col}: {n} outliers')

# Note: which columns have the most outliers, and are they real story events or data errors?

<a name="skip-code-30"></a>

**Expected output:** A list of columns with their outlier counts. `outliers 1` and `outliers 2` should appear near the top. In your story context, decide whether each outlier represents a genuine extreme event or a measurement error — your choice determines the treatment strategy in Part 4.

<nav aria-label="Return navigation for code cell 30">
<a href="#read-code-30" aria-label="Go back and read code cell 30">&#8617; Go back and read the code cell</a>
</nav>

## 10 Essential EDA Visualizations

The code templates below cover the ten most useful EDA charts. For each one: copy it into a new code cell below its description, replace the placeholder column names with your story variable names, run it, and add a one-sentence observation. Add a Markdown cell after each chart describing what it shows — this serves as the chart's alt text.

**Run this prerequisite cell first:**

<a name="read-code-31"></a>

**Cell 31 — EDA Prerequisites**

This cell imports the visualization libraries and converts the `date` column to datetime format. It must run before any of the 10 visualization templates below.

<nav aria-label="Code cell 31 navigation">
<a href="#skip-code-31" aria-label="Skip code cell 31 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df['date'] = pd.to_datetime(df['date'])
print('EDA libraries loaded and date column converted.')

<a name="skip-code-31"></a>

**Expected output:** `EDA libraries loaded and date column converted.` If you see a parsing error on the date column, check that it loaded correctly from CSV.

<nav aria-label="Return navigation for code cell 31">
<a href="#read-code-31" aria-label="Go back and read code cell 31">&#8617; Go back and read the code cell</a>
</nav>

### Visualization Templates

Copy each template into a new code cell below, replace column names with your story names, and run.

**1. Histogram and KDE — Univariate Distribution**
```python
plt.figure(figsize=(10, 6))
sns.histplot(df['target'], kde=True, color='blue')
plt.title('Distribution of Target Variable')
plt.xlabel('Target Value')
plt.ylabel('Frequency')
plt.show()
```

**2. Box Plot — Outliers and Quartiles**
```python
plt.figure(figsize=(10, 6))
sns.boxplot(x='class', y='informative_1', data=df, palette='Set2')
plt.title('Informative_1 Distribution by Class')
plt.show()
```

**3. Count Plot — Categorical Frequency**
```python
plt.figure(figsize=(10, 6))
sns.countplot(x='categorical_1', data=df, order=df['categorical_1'].value_counts().index)
plt.title('Frequency of Categorical_1 Levels')
plt.xticks(rotation=45)
plt.show()
```

**4. Scatter Plot — Numerical Relationships**
```python
plt.figure(figsize=(10, 6))
sns.scatterplot(x='informative_1', y='target', hue='class', data=df, alpha=0.6)
plt.title('Informative_1 vs Target (Colored by Class)')
plt.show()
```

**5. Correlation Heatmap — Multicollinearity**
```python
plt.figure(figsize=(12, 10))
numeric_df = df.select_dtypes(include=['float64', 'int64'])
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix Heatmap')
plt.show()
```

**6. Pair Plot — Multivariate Overview**
```python
cols = ['informative_1', 'informative_2', 'target', 'class']
sns.pairplot(df[cols], hue='class', diag_kind='kde')
plt.show()
```

**7. Violin Plot — Density by Category**
```python
plt.figure(figsize=(10, 6))
sns.violinplot(x='pd qcut1', y='target', data=df, inner='quartile')
plt.title('Target Density by pd qcut1')
plt.show()
```

**8. Bar Plot — Aggregated Statistics**
```python
plt.figure(figsize=(10, 6))
sns.barplot(x='class', y='correlated w target 1', data=df, estimator='mean', errorbar='sd')
plt.title('Mean of Correlated Feature per Class (with Std Dev)')
plt.show()
```

**9. Line Plot — Time Series Trend**
```python
plt.figure(figsize=(12, 6))
df_time = df.sort_values('date')
sns.lineplot(x='date', y='target', data=df_time)
plt.title('Target Variable Trend Over Time')
plt.xticks(rotation=45)
plt.show()
```

**10. Joint Plot — Bivariate and Marginal Distributions**
```python
sns.jointplot(x='multicollinearity 1', y='multicollinearity 2', data=df, kind='reg', color='purple')
plt.suptitle('Joint Plot of Multicollinear Features', y=1.02)
plt.show()
```

<a name="read-code-32"></a>

**Cell 32 — Save EDA Dataset**

This cell saves the current DataFrame (with the date column now parsed as datetime) to `data science fiction iv pt 2.csv`. Part 4 loads from this file. Do not skip.

<nav aria-label="Code cell 32 navigation">
<a href="#skip-code-32" aria-label="Skip code cell 32 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
df.to_csv('data science fiction iv pt 2.csv', index=False)
print('EDA dataset saved.')

<a name="skip-code-32"></a>

**Expected output:** `EDA dataset saved.`

<nav aria-label="Return navigation for code cell 32">
<a href="#read-code-32" aria-label="Go back and read code cell 32">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 4: Data Preparation

In this part you will systematically clean the dataset by removing or correcting the problems that were deliberately introduced in Part 1. Work through each section **in order** — some steps depend on the previous ones being complete.

**Cleaning checklist:**

1. Constants — drop columns where every value is identical
2. Quasi-constants — drop columns where one value dominates (>98%)
3. Duplicate rows — remove exact duplicate records
4. Duplicate features — remove columns that are identical to another column
5. Missing data — impute or drop columns with nulls
6. Scaling — apply the correct scaler to columns that need it
7. Outliers — choose a treatment strategy for each outlier column

Use your story variable names throughout.

<a name="read-code-33"></a>

**Cell 33 — Load the EDA Dataset**

This cell loads the dataset saved at the end of Part 3. Run it before any cleaning steps.

<nav aria-label="Code cell 33 navigation">
<a href="#skip-code-33" aria-label="Skip code cell 33 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# import pandas as pd

# df = pd.read_csv('data science fiction iv pt 2.csv')
# print(df.shape)
# print(df.info())
# df.head()

<a name="skip-code-33"></a>

**Expected output:** Shape of (1010, ~38). If the shape differs significantly, check that you saved correctly at the end of Part 3.

<nav aria-label="Return navigation for code cell 33">
<a href="#read-code-33" aria-label="Go back and read code cell 33">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-34"></a>

**Cell 34 — Remove Constant Columns**

Columns where every row has the same value carry zero information. Use `df.nunique()` to find columns with only 1 unique value, store the list, and drop them.

<nav aria-label="Code cell 34 navigation">
<a href="#skip-code-34" aria-label="Skip code cell 34 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# CONSTANTS
# Hint: df.nunique()[df.nunique() == 1].index gives you the constant columns.

# YOUR CODE HERE


# How many constant columns did you find and remove? Note it here:
# Answer:

<a name="skip-code-34"></a>

**Expected result:** 2 constant columns removed (`constant_1` and `constant_2`). Run `df.shape` to confirm the column count decreased.

<nav aria-label="Return navigation for code cell 34">
<a href="#read-code-34" aria-label="Go back and read code cell 34">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-35"></a>

**Cell 35 — Remove Quasi-Constant Columns**

Columns where one value appears in over 98% of rows add almost no predictive signal. For each categorical column, compute `value_counts(normalize=True)` and flag columns where the top proportion exceeds 0.98.

<nav aria-label="Code cell 35 navigation">
<a href="#skip-code-35" aria-label="Skip code cell 35 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# QUASI-CONSTANTS
# Hint: series.value_counts(normalize=True).iloc[0] gives the top proportion.

# YOUR CODE HERE


# How many quasi-constant columns did you find and remove? Note it here:
# Answer:

<a name="skip-code-35"></a>

**Expected result:** 2 quasi-constant columns removed (`semi_constant_1` and `semi_constant_2`). These had only 1% variation and are useless as predictors.

<nav aria-label="Return navigation for code cell 35">
<a href="#read-code-35" aria-label="Go back and read code cell 35">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-36"></a>

**Cell 36 — Remove Duplicate Rows**

The 10 duplicate rows added in Part 1 must be removed. Use `df.duplicated().sum()` to count them, `df.drop_duplicates()` to remove them, and `df.reset_index(drop=True)` to reset the index.

<nav aria-label="Code cell 36 navigation">
<a href="#skip-code-36" aria-label="Skip code cell 36 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# DUPLICATE ROWS
# YOUR CODE HERE


# How many duplicate rows were removed? Note it here:
# Answer:

<a name="skip-code-36"></a>

**Expected result:** 10 duplicate rows removed. Shape should now show 1,000 rows.

<nav aria-label="Return navigation for code cell 36">
<a href="#read-code-36" aria-label="Go back and read code cell 36">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-37"></a>

**Cell 37 — Remove Duplicate Features**

Columns that are identical to another column are redundant. Use `df.T.duplicated()` to find them and drop one from each duplicate pair.

<nav aria-label="Code cell 37 navigation">
<a href="#skip-code-37" aria-label="Skip code cell 37 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# DUPLICATE FEATURES
# Hint: df.columns[df.T.duplicated()].tolist() gives you the duplicates to drop.

# YOUR CODE HERE


# Which columns were duplicates? Note them here:
# Answer:

<a name="skip-code-37"></a>

**Expected result:** `duplicate_1` and `duplicate_2` removed. These were exact copies of `informative_1` and `informative_2`.

<nav aria-label="Return navigation for code cell 37">
<a href="#read-code-37" aria-label="Go back and read code cell 37">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-38"></a>

**Cell 38 — Handle Missing Data**

Work through nulls column by column. First run `df.isnull().sum() / len(df) * 100` to see percentage missing per column. Drop columns with over 60% missing. For the rest: impute numerical columns with mean or median, and categorical columns with mode or a placeholder like 'Unknown'.

<nav aria-label="Code cell 38 navigation">
<a href="#skip-code-38" aria-label="Skip code cell 38 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# MISSING DATA
# Step 1: Check % missing per column
# print((df.isnull().sum() / len(df) * 100).sort_values(ascending=False))

# Step 2: Drop columns exceeding your threshold (suggest 60%)
# YOUR CODE HERE

# Step 3: Impute remaining nulls
# YOUR CODE HERE

# Verify: df.isnull().sum().sum() should return 0 after imputation
# print('Remaining nulls:', df.isnull().sum().sum())

<a name="skip-code-38"></a>

**Expected result:** All null values resolved. `df.isnull().sum().sum()` should return 0. Note which columns you dropped (too many nulls) and which you imputed.

<nav aria-label="Return navigation for code cell 38">
<a href="#read-code-38" aria-label="Go back and read code cell 38">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-39"></a>

**Cell 39 — Apply Scaling**

Two columns need scaling before modeling: the one generated by `gen_standard_scaling` (apply StandardScaler: centers to mean=0, std=1) and the one from `gen_minmax_scaling` (apply MinMaxScaler: compresses to [0, 1]). Refer to your story mapping to identify which columns these are.

<nav aria-label="Code cell 39 navigation">
<a href="#skip-code-39" aria-label="Skip code cell 39 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# SCALING
# from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Standard Scaler — use for the gen_standard_scaling column
# YOUR CODE HERE

# MinMax Scaler — use for the gen_minmax_scaling column
# YOUR CODE HERE

# Note: In a production pipeline (Part 6), scalers should be fit on training data only.
# We apply them to the full df here for simplicity.

<a name="skip-code-39"></a>

**Expected result:** The two scaled columns should now have values comparable in magnitude to the other numerical features. StandardScaler output centers near 0; MinMaxScaler output falls between 0 and 1.

<nav aria-label="Return navigation for code cell 39">
<a href="#read-code-39" aria-label="Go back and read code cell 39">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-40"></a>

**Cell 40 — Treat Outliers**

Choose a treatment strategy for each outlier-heavy column identified in Part 3. For each one, comment on whether the outlier represents a real extreme event in your story or a data error, and apply the appropriate strategy:

- **Winsorization:** Cap at the 5th/95th percentile — preserves rows, limits extreme influence
- **Trimming:** Remove rows beyond ±3 std dev — use only for clear data errors
- **Log transformation:** `np.log1p()` — use for right-skewed data with real extreme values

<nav aria-label="Code cell 40 navigation">
<a href="#skip-code-40" aria-label="Skip code cell 40 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# OUTLIERS
# For each outlier column, choose a strategy and add a comment explaining your reasoning.

# Example — Winsorization:
# col = 'your_outlier_column_story_name'
# df[col] = df[col].clip(lower=df[col].quantile(0.05), upper=df[col].quantile(0.95))
# Reasoning: (explain whether this outlier is a real event or a data error in your story)

# YOUR CODE HERE for each outlier column

<a name="skip-code-40"></a>

**Expected result:** Outlier columns with reduced extreme values. Run `df.boxplot()` on each treated column to confirm the outliers are handled. Your reasoning comments are part of the grade.

<nav aria-label="Return navigation for code cell 40">
<a href="#read-code-40" aria-label="Go back and read code cell 40">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-41"></a>

**Cell 41 — Re-identify Variable Types After Cleaning**

After cleaning, the column set has changed. This cell re-runs the type identification so Part 5 has an accurate view of what remains.

<nav aria-label="Code cell 41 navigation">
<a href="#skip-code-41" aria-label="Skip code cell 41 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# df_numerical = df.select_dtypes(include='number').columns
# df_object = df.select_dtypes(include=['object']).columns
# df_discrete = df.select_dtypes(include=['category']).columns
# df_categorical_features = df.select_dtypes(include=['category', 'object']).columns
# print('Numerical:', list(df_numerical))
# print('Object:', list(df_object))
# print('Category:', list(df_discrete))

<a name="skip-code-41"></a>

**Expected output:** Updated column lists reflecting the cleaned dataset. The constant, quasi-constant, duplicate, and dropped high-null columns should no longer appear.

<nav aria-label="Return navigation for code cell 41">
<a href="#read-code-41" aria-label="Go back and read code cell 41">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 5: Feature Engineering

In this part you will:
1. **Create derived variables** — new columns computed from existing ones.
2. **Encode categorical variables** — convert text and category columns into numbers the model can use.

After this part, every column in `df` (except `class`) must be numerical. The final cell saves the result for Part 6.

## Derived Variables

A derived variable captures a relationship that raw columns may not express on their own. Create at least **two** derived variables that make sense in your historical story.

| Technique | Example | When to Use |
| :--- | :--- | :--- |
| Ratio | `df['a'] / (df['b'] + 1e-9)` | When the relationship between two quantities matters more than their individual values |
| Interaction | `df['a'] * df['b']` | When two features jointly affect the outcome |
| Binning | `pd.cut(df['col'], bins=3, labels=['Low','Med','High'])` | To convert continuous to ordered category |
| Date component | `pd.to_datetime(df['date']).dt.year` | To extract meaningful temporal units |

<a name="read-code-42"></a>

**Cell 42 — Create Derived Variables**

Write at least two new computed columns using your story variable names. For each one, add a comment explaining: which columns it combines, what it represents in your historical story, and why it might help predict the outcome.

<nav aria-label="Code cell 42 navigation">
<a href="#skip-code-42" aria-label="Skip code cell 42 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# DERIVED VARIABLES — create at least two

# Example structure:
# df['your_derived_name_1'] = df['col_a'] / (df['col_b'] + 1e-9)
# Comment: ratio of X to Y — in the story this represents...

# df['your_derived_name_2'] = df['col_a'] * df['col_b']
# Comment: interaction of X and Y — this combination is important because...

# YOUR CODE HERE

<a name="skip-code-42"></a>

**Expected result:** At least two new columns in `df`. Print `df.shape` to confirm they were added. Your story-based comments are part of the grade.

<nav aria-label="Return navigation for code cell 42">
<a href="#read-code-42" aria-label="Go back and read code cell 42">&#8617; Go back and read the code cell</a>
</nav>

## Categorical Encoding

All remaining object and category columns must be converted to numbers before modeling.

| Situation | Recommended Encoding |
| :--- | :--- |
| 2 unique values (binary) | `df['col'].map({'Label_A': 0, 'Label_B': 1})` |
| 3–10 values, no natural order | `pd.get_dummies(df, columns=['col'], drop_first=True)` |
| 3+ values with natural order | `OrdinalEncoder` with defined category list |
| 10+ values or not useful | Drop the column |

<a name="read-code-43"></a>

**Cell 43 — Encode Categorical Variables**

Apply the appropriate encoding to every remaining object and category column. Drop columns that cannot be meaningfully encoded (raw name fields, free-text, zip codes used as identifiers). Leave a comment for each column noting which method you chose and why.

<nav aria-label="Code cell 43 navigation">
<a href="#skip-code-43" aria-label="Skip code cell 43 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# CATEGORICAL ENCODING

# Binary example:
# df['col'] = df['col'].map({'Label_A': 0, 'Label_B': 1})

# One-hot example:
# df = pd.get_dummies(df, columns=['col'], drop_first=True)

# Ordinal example:
# from sklearn.preprocessing import OrdinalEncoder
# enc = OrdinalEncoder(categories=[['Low', 'Med', 'High']])
# df[['col']] = enc.fit_transform(df[['col']])

# Drop example:
# df.drop(columns=['name_col', 'free_text_col'], inplace=True)

# YOUR CODE HERE

<a name="skip-code-43"></a>

**Expected result:** No object or category columns remaining (except `class`). Run the verification cell below to confirm.

<nav aria-label="Return navigation for code cell 43">
<a href="#read-code-43" aria-label="Go back and read code cell 43">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-44"></a>

**Cell 44 — Verify All Columns Are Numerical**

This cell checks that every column except `class` is now a numeric type. If any object columns remain, return to the encoding cell and address them before continuing.

<nav aria-label="Code cell 44 navigation">
<a href="#skip-code-44" aria-label="Skip code cell 44 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# non_numeric = df.drop('class', axis=1).select_dtypes(exclude='number').columns.tolist()
# if non_numeric:
#     print('These columns still need encoding or removal:', non_numeric)
# else:
#     print('All columns are numerical. Ready for pipeline and feature selection.')
# df.info()

<a name="skip-code-44"></a>

**Expected output:** `All columns are numerical. Ready for pipeline and feature selection.` If non-numeric columns are listed, address each one in Cell 43 before proceeding.

<nav aria-label="Return navigation for code cell 44">
<a href="#read-code-44" aria-label="Go back and read code cell 44">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-45"></a>

**Cell 45 — Save the Encoded Dataset**

This cell saves the fully encoded DataFrame to `data science fiction iv pt 3.csv`. Part 6 (Pipelines) and Part 7 (Feature Selection) both load from this file.

<nav aria-label="Code cell 45 navigation">
<a href="#skip-code-45" aria-label="Skip code cell 45 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# df.to_csv('data science fiction iv pt 3.csv', index=False)
# print('Encoded dataset saved.')

<a name="skip-code-45"></a>

**Expected output:** `Encoded dataset saved.`

<nav aria-label="Return navigation for code cell 45">
<a href="#read-code-45" aria-label="Go back and read code cell 45">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 6: Pipelines

In Parts 4 and 5 you cleaned and encoded your data step by step in individual cells. In a real production environment, every one of those steps must run automatically every time new data arrives — without human intervention and without the risk of applying steps in the wrong order or fitting a scaler on test data.

A **scikit-learn Pipeline** solves this by chaining preprocessing and modeling steps into a single object that can be fit once and applied consistently to any new data.

## Why Pipelines Matter

| Problem Without Pipelines | How Pipelines Solve It |
| :--- | :--- |
| Scalers accidentally fit on test data (data leakage) | Pipeline fits only on `X_train`, transforms `X_test` automatically |
| Steps applied in wrong order across notebooks | Sequential steps are locked in place |
| Messy, hard-to-reproduce code | Single object that can be saved, shared, and reloaded |
| Manual re-application every time new data arrives | `pipeline.transform(new_data)` applies all steps at once |

## Pipeline Architecture

```
Raw Data
   |
   ▼
ColumnTransformer  ←── Different transformers applied to different column types
   ├── Numerical columns:   SimpleImputer(median) → StandardScaler
   └── Categorical columns: SimpleImputer(mode)   → OneHotEncoder
   |
   ▼
LogisticRegression  ←── Model receives clean, scaled, encoded data
```

The key class is `ColumnTransformer`, which applies different transformers to different subsets of columns simultaneously.

<a name="read-code-46"></a>

**Cell 46 — Load the Encoded Dataset and Prepare Train/Test Split**

This cell loads the dataset saved at the end of Part 5, then creates a stratified train/test split. The features (`X`) and target (`y`) are separated before building the pipeline.

<nav aria-label="Code cell 46 navigation">
<a href="#skip-code-46" aria-label="Skip code cell 46 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('data science fiction iv pt 3.csv')

X = df.drop('class', axis=1)
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)

print('X_train shape:', X_train.shape)
print('X_test shape: ', X_test.shape)
print('Class balance in y_train:', y_train.value_counts().to_dict())

<a name="skip-code-46"></a>

**Expected output:** Training set with ~700 rows, test set with ~300 rows. The class balance should be approximately equal between train and test sets (stratify ensures this). If you see a shape mismatch, confirm that Cell 45 saved correctly.

<nav aria-label="Return navigation for code cell 46">
<a href="#read-code-46" aria-label="Go back and read code cell 46">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-47"></a>

**Cell 47 — Identify Column Types for the Pipeline**

A ColumnTransformer needs to know which columns are numerical and which are categorical. This cell identifies them so you can pass the correct lists to the transformer.

<nav aria-label="Code cell 47 navigation">
<a href="#skip-code-47" aria-label="Skip code cell 47 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Identify numerical and categorical columns from X_train
numerical_cols = X_train.select_dtypes(include='number').columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Numerical columns ({len(numerical_cols)}):', numerical_cols)
print(f'\nCategorical columns ({len(categorical_cols)}):', categorical_cols)

<a name="skip-code-47"></a>

**Expected output:** Two lists of column names. Because you encoded all categoricals in Part 5, you may find `categorical_cols` is empty — that is expected and means your pipeline will only need a numerical transformer.

<nav aria-label="Return navigation for code cell 47">
<a href="#read-code-47" aria-label="Go back and read code cell 47">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-48"></a>

**Cell 48 — Build the Preprocessing Pipeline**

This cell assembles a full preprocessing pipeline using `ColumnTransformer`. For numerical columns: missing values are imputed with the median, then StandardScaler is applied. For categorical columns (if any remain): missing values are imputed with the most frequent value, then OneHotEncoder converts them to numbers. The two transformers are combined into a single `preprocessor` object.

<nav aria-label="Code cell 48 navigation">
<a href="#skip-code-48" aria-label="Skip code cell 48 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Numerical transformer: impute median -> standard scale
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical transformer: impute mode -> one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
], remainder='drop')  # Drop any columns not explicitly assigned

print('Preprocessor pipeline assembled.')
print(preprocessor)

<a name="skip-code-48"></a>

**Expected output:** A text description of the `ColumnTransformer` object showing its two sub-pipelines (numerical and categorical). This object has not run yet — it is fit in the next cell.

<nav aria-label="Return navigation for code cell 48">
<a href="#read-code-48" aria-label="Go back and read code cell 48">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-49"></a>

**Cell 49 — Fit the Pipeline and Transform the Data**

This cell fits the preprocessor **only on the training data** and transforms both sets. This is the critical distinction: the scaler learns the mean and std from training data only, then applies those values to the test set. This prevents data leakage.

<nav aria-label="Code cell 49 navigation">
<a href="#skip-code-49" aria-label="Skip code cell 49 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import numpy as np

# Fit on training data only — never on test data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print('Training set shape after preprocessing:', X_train_processed.shape)
print('Test set shape after preprocessing:    ', X_test_processed.shape)

# Check that no nulls remain
print('Nulls in processed training set:', np.isnan(X_train_processed).sum())
print('Nulls in processed test set:    ', np.isnan(X_test_processed).sum())

<a name="skip-code-49"></a>

**Expected output:** Both processed sets should have 0 nulls. The number of columns may differ from the input if OneHotEncoder expanded any categoricals. The key result is that `preprocessor.fit()` was called on `X_train` only.

<nav aria-label="Return navigation for code cell 49">
<a href="#read-code-49" aria-label="Go back and read code cell 49">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-50"></a>

**Cell 50 — Build a Full Modeling Pipeline**

This cell combines the preprocessor and a logistic regression model into a single end-to-end pipeline. This is the pattern used in production: you pass raw data in, and the pipeline handles preprocessing and prediction automatically.

<nav aria-label="Code cell 50 navigation">
<a href="#skip-code-50" aria-label="Skip code cell 50 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver='liblinear', random_state=42))
])

# Fit the entire pipeline on raw training data
full_pipeline.fit(X_train, y_train)

# Evaluate on raw test data — pipeline preprocesses automatically
pipeline_train_score = full_pipeline.score(X_train, y_train)
pipeline_test_score = full_pipeline.score(X_test, y_test)

print(f'Pipeline Training Accuracy: {pipeline_train_score:.4f}')
print(f'Pipeline Test Accuracy:     {pipeline_test_score:.4f}')

<a name="skip-code-50"></a>

**Expected output:** Training and test accuracy scores. A large gap between the two (e.g., training 0.90, test 0.65) suggests overfitting — you will diagnose this formally with bias-variance decomposition in Part 8. The advantage of the full pipeline is that `full_pipeline.predict(new_raw_data)` now works on any new, unprocessed data.

<nav aria-label="Return navigation for code cell 50">
<a href="#read-code-50" aria-label="Go back and read code cell 50">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-51"></a>

**Cell 51 — Inspect Pipeline Steps**

This cell shows the internal structure of the full pipeline and lists the named steps so you know how to access each component programmatically.

<nav aria-label="Code cell 51 navigation">
<a href="#skip-code-51" aria-label="Skip code cell 51 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Inspect the pipeline structure
print('Pipeline steps:')
for name, step in full_pipeline.steps:
    print(f'  {name}: {type(step).__name__}')

# Access the preprocessor step
print('\nPreprocessor transformers:')
for name, transformer, cols in full_pipeline.named_steps['preprocessor'].transformers_:
    print(f'  {name}: {type(transformer).__name__} on {len(cols)} columns')

# Access model coefficients
classifier = full_pipeline.named_steps['classifier']
print(f'\nModel coefficients shape: {classifier.coef_.shape}')

<a name="skip-code-51"></a>

**Expected output:** A structured listing of pipeline steps and transformer details. The coefficient shape will show (1, N) where N is the number of processed features. This cell demonstrates that every component of a pipeline is individually accessible for inspection and debugging.

<nav aria-label="Return navigation for code cell 51">
<a href="#read-code-51" aria-label="Go back and read code cell 51">&#8617; Go back and read the code cell</a>
</nav>

### ✏️ Pipeline Reflection

Edit this Markdown cell and answer the following questions in complete sentences:

1. **Data leakage prevention:** In your own words, explain why the scaler must be fit on training data only. What would happen to your test accuracy if you fit it on the full dataset first?

2. **Reproducibility:** Describe one scenario in your historical catastrophe story where a pipeline would be essential — for example, if new archaeological data arrived monthly or new health records were received daily.

3. **Pipeline vs. manual steps:** In this notebook you cleaned data manually in Parts 4 and 5 and then built a pipeline in Part 6. In a real project, which approach would you use and why?

---

*Write your response here.*

---

---
# Part 7: Feature Selection

Feature selection identifies which variables actually help your model generalize. With a large dataset you are looking to reduce noise and multicollinearity before the final model. You will apply five selection methods and compare their recommendations.

**Feature selection categories:**

| Category | Method | Description |
| :--- | :--- | :--- |
| Filter | CorrWith, Mutual Information, SelectKBest | Score features independently of the model |
| Wrapper | RFE | Train the model on feature subsets, keep the best |
| Embedded | SelectFromModel, Random Forest | Selection happens during model training |

<a name="read-code-52"></a>

**Cell 52 — Load the Encoded Dataset for Feature Selection**

This cell reloads the encoded dataset saved in Part 5 and creates a train/test split for feature selection.

<nav aria-label="Code cell 52 navigation">
<a href="#skip-code-52" aria-label="Skip code cell 52 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# import pandas as pd
# from sklearn.model_selection import train_test_split

# df = pd.read_csv('data science fiction iv pt 3.csv')
# X_train, X_test, y_train, y_test = train_test_split(
#     df.drop('class', axis=1), df['class'],
#     test_size=0.3, random_state=42)
# print('X_train:', X_train.shape, '  X_test:', X_test.shape)

<a name="skip-code-52"></a>

**Expected output:** Training and test set shapes. These splits are used consistently across all five selection methods below.

<nav aria-label="Return navigation for code cell 52">
<a href="#read-code-52" aria-label="Go back and read code cell 52">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-53"></a>

**Cell 53 — CorrWith: Correlation with the Target (Filter Method)**

This cell computes the Pearson correlation between each feature and the `class` target. Higher absolute correlation suggests a stronger linear relationship with the outcome. Note the top 5 features — you will compare across all five methods at the end.

<nav aria-label="Code cell 53 navigation">
<a href="#skip-code-53" aria-label="Skip code cell 53 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# corr_with_target = X_train.corrwith(y_train).abs().sort_values(ascending=False)
# print(corr_with_target)

# Top 5 features by CorrWith:
# 1.
# 2.
# 3.
# 4.
# 5.

<a name="skip-code-53"></a>

**Expected output:** A ranked list of features by absolute Pearson correlation with `class`. Your `informative_1`, `informative_2`, and `correlated w target` columns should rank highly.

<nav aria-label="Return navigation for code cell 53">
<a href="#read-code-53" aria-label="Go back and read code cell 53">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-54"></a>

**Cell 54 — Mutual Information (Filter Method)**

Mutual Information measures how much knowing a feature reduces uncertainty about the class label. Unlike Pearson correlation, it captures non-linear relationships. Compare your top 5 with the CorrWith results.

<nav aria-label="Code cell 54 navigation">
<a href="#skip-code-54" aria-label="Skip code cell 54 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# from sklearn.feature_selection import mutual_info_classif
# import matplotlib.pyplot as plt

# mi = mutual_info_classif(X_train, y_train)
# mi = pd.Series(mi)
# mi.index = X_train.columns
# mi.sort_values(ascending=False).plot.bar()
# plt.ylabel('Mutual Information')
# plt.tight_layout()
# plt.show()

# mi_keepers = mi.sort_values(ascending=False).index[:5]
# print(mi_keepers)

<a name="skip-code-54"></a>

**Expected output:** A bar chart followed by the top 5 feature names. Features with MI score of 0 carry no information about the target and are strong drop candidates.

**Accessibility note:** Add a Markdown cell below describing what the bar chart shows — which features have the highest information gain and what that means for your story.

<nav aria-label="Return navigation for code cell 54">
<a href="#read-code-54" aria-label="Go back and read code cell 54">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-55"></a>

**Cell 55 — SelectKBest (Filter Method)**

SelectKBest applies an F-statistic test (`f_classif`) to score each feature and returns the top k. Compare results to the two filter methods above.

<nav aria-label="Code cell 55 navigation">
<a href="#skip-code-55" aria-label="Skip code cell 55 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# from sklearn.feature_selection import SelectKBest, f_classif

# selector = SelectKBest(f_classif, k=5)
# selector.fit(X_train, y_train)
# kb_keepers = X_train.columns.values[selector.get_support()]
# print('SelectKBest top 5:', kb_keepers)

<a name="skip-code-55"></a>

**Expected output:** A list of 5 feature names selected by the F-test. These should overlap substantially with CorrWith and Mutual Information — agreement across methods is a strong signal.

<nav aria-label="Return navigation for code cell 55">
<a href="#read-code-55" aria-label="Go back and read code cell 55">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-56"></a>

**Cell 56 — SelectFromModel with Logistic Regression (Embedded Method)**

This cell trains a regularized logistic regression and keeps only features with non-negligible coefficients. Because selection happens during training, this is an embedded method. StandardScaler is applied first since logistic regression is sensitive to feature scale.

<nav aria-label="Code cell 56 navigation">
<a href="#skip-code-56" aria-label="Skip code cell 56 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.feature_selection import SelectFromModel
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X_train)

# selections = SelectFromModel(estimator=LogisticRegression()).fit(X_scaled, y_train)
# mt_keepers = X_train.columns.values[selections.get_support()]
# print('SelectFromModel keepers:', mt_keepers)

<a name="skip-code-56"></a>

**Expected output:** A list of feature names kept by the regularized model. L1 or L2 regularization shrinks unimportant coefficients toward zero, effectively performing automatic feature selection.

<nav aria-label="Return navigation for code cell 56">
<a href="#read-code-56" aria-label="Go back and read code cell 56">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-57"></a>

**Cell 57 — Recursive Feature Elimination (Wrapper Method)**

RFE trains the model repeatedly, removing the weakest feature each iteration, until only the target number of features remain. It is the most thorough of the five methods. This may take a few seconds to run.

<nav aria-label="Code cell 57 navigation">
<a href="#skip-code-57" aria-label="Skip code cell 57 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# from sklearn.feature_selection import RFE
# from sklearn.linear_model import LogisticRegression

# estimator = LogisticRegression()
# selector = RFE(estimator, n_features_to_select=5)
# selector.fit(X_scaled, y_train)
# rf_keepers = X_train.columns.values[selector.get_support()]
# print('RFE top 5:', rf_keepers)

<a name="skip-code-57"></a>

**Expected output:** A list of 5 features surviving iterative elimination. RFE considers feature interactions during elimination, so its results can differ from pure filter methods.

<nav aria-label="Return navigation for code cell 57">
<a href="#read-code-57" aria-label="Go back and read code cell 57">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-58"></a>

**Cell 58 — Random Forest Feature Importance (Embedded Method)**

Random Forest ranks features by how much they reduce impurity across all trees. It handles non-linear relationships well and is less sensitive to scaling than logistic regression.

<nav aria-label="Code cell 58 navigation">
<a href="#skip-code-58" aria-label="Skip code cell 58 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.feature_selection import SelectFromModel

# selects = SelectFromModel(
#     RandomForestClassifier(n_estimators=100, random_state=42), max_features=4)
# selects.fit(X_train, y_train)
# rfi = X_train.columns[(selects.get_support())]
# print('Random Forest top features:', rfi.tolist())

<a name="skip-code-58"></a>

**Expected output:** Up to 4 feature names ranked by impurity reduction. Compare with the four previous methods in your summary table below.

<nav aria-label="Return navigation for code cell 58">
<a href="#read-code-58" aria-label="Go back and read code cell 58">&#8617; Go back and read the code cell</a>
</nav>

## Build Your Final Feature List

Complete this summary table before running the final two cells. Also revisit your VIF results from Part 3 and your outlier notes from Part 4.

| Feature | CorrWith | MI | SelectKBest | SelectFromModel | RFE | Random Forest | Total |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| *feature name* | ✓ | ✓ | | ✓ | | ✓ | 4 |

Features appearing in **3 or more methods** are strong candidates. Also consider whether each feature makes narrative sense as a predictor of your historical catastrophe. **Aim for 5–8 features.** Too few → underfitting. Too many → overfitting.

<a name="read-code-59"></a>

**Cell 59 — Define Your Final Feature List**

Based on the summary table above, list the features you will use for modeling. Add a comment for each one noting how many methods selected it and why it makes sense in your story.

<nav aria-label="Code cell 59 navigation">
<a href="#skip-code-59" aria-label="Skip code cell 59 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# features_to_model = [
#     # 'your_feature_1',  # appeared in N methods — story reason
#     # 'your_feature_2',  # appeared in N methods — story reason
#     # add more as needed (aim for 5-8)
# ]

# print(f'{len(features_to_model)} features selected:', features_to_model)

<a name="skip-code-59"></a>

**Expected output:** A list of 5–8 feature names and their count. This list is your `features_to_model` and is used in Part 8.

<nav aria-label="Return navigation for code cell 59">
<a href="#read-code-59" aria-label="Go back and read code cell 59">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-60"></a>

**Cell 60 — Filter Train and Test Sets to Selected Features**

This cell restricts `X_train` and `X_test` to only the features you selected. The final shapes confirm how many features enter the model.

<nav aria-label="Code cell 60 navigation">
<a href="#skip-code-60" aria-label="Skip code cell 60 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# X_train = X_train[features_to_model]
# X_test = X_test[features_to_model]
# print('Final X_train shape:', X_train.shape)
# print('Final X_test shape:', X_test.shape)

<a name="skip-code-60"></a>

**Expected output:** Shapes with the column count matching your `features_to_model` list length. You are now ready for modeling.

<nav aria-label="Return navigation for code cell 60">
<a href="#read-code-60" aria-label="Go back and read code cell 60">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 8: Modeling and Evaluation

You have cleaned, engineered, pipelined, and selected your features. Now you will train a logistic regression and evaluate it thoroughly.

**This part has both code tasks and written explanation tasks.** The explanation cells are graded — write in complete sentences and connect your answers to your story.

<a name="read-code-61"></a>

**Cell 61 — Train the Logistic Regression Model**

This cell fits a logistic regression model on your selected training features and reports accuracy on both the training and test sets. A large gap between training and test accuracy indicates overfitting.

<nav aria-label="Code cell 61 navigation">
<a href="#skip-code-61" aria-label="Skip code cell 61 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import confusion_matrix, accuracy_score

# model = LogisticRegression(solver='liblinear', random_state=42)
# model.fit(X_train, y_train)
# predictions = model.predict(X_test)

# train_accuracy = model.score(X_train, y_train)
# test_accuracy = model.score(X_test, y_test)
# print(f'Training Accuracy: {train_accuracy:.4f}')
# print(f'Testing Accuracy:  {test_accuracy:.4f}')

<a name="skip-code-61"></a>

**Expected output:** Two accuracy scores. A well-calibrated model will show training and test accuracy within 5–10% of each other. Note both values — you will reference the gap in the bias-variance explanation cell.

<nav aria-label="Return navigation for code cell 61">
<a href="#read-code-61" aria-label="Go back and read code cell 61">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-62"></a>

**Cell 62 — Confusion Matrix**

This cell extracts the four confusion matrix values: True Negatives (TN), False Positives (FP), False Negatives (FN), and True Positives (TP). These are the foundation for all classification metrics.

<nav aria-label="Code cell 62 navigation">
<a href="#skip-code-62" aria-label="Skip code cell 62 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# acc = accuracy_score(y_test, predictions)
# print(f'Overall Accuracy: {acc:.4f}')

# cm = confusion_matrix(y_test, predictions)
# tn, fp, fn, tp = cm.ravel()
# print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}')
# print('\nFull Confusion Matrix:')
# print(cm)

<a name="skip-code-62"></a>

**Expected output:** Overall accuracy and a breakdown of TN, FP, FN, TP. Translate these into story terms in the written explanation cell below.

<nav aria-label="Return navigation for code cell 62">
<a href="#read-code-62" aria-label="Go back and read code cell 62">&#8617; Go back and read the code cell</a>
</nav>

### ✏️ Explanation — Confusion Matrix

Answer the following questions in complete sentences. Connect each answer to your historical story.

1. How many True Positives and True Negatives did your model produce? What do these represent in your historical scenario?

2. Which error type did your model make more of — False Positives or False Negatives? Translate this into story terms (for example: 'The model raised the alarm when there was no plague X times, and missed an actual plague outbreak Y times.').

3. Referring back to Part 2's hypothesis framework: did your model make more Type I errors (false alarms) or Type II errors (missed catastrophes)? Given the stakes of your historical scenario, which error type is more costly, and why?

---
*Write your response here.*

---

<a name="read-code-63"></a>

**Cell 63 — Classification Report**

This cell prints the full classification report showing precision, recall, F1-score, and support for each class. Use these values to answer the explanation questions below.

<nav aria-label="Code cell 63 navigation">
<a href="#skip-code-63" aria-label="Skip code cell 63 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# print(classification_report(y_test, predictions))

<a name="skip-code-63"></a>

**Expected output:** A formatted table with precision, recall, F1, and support per class, followed by macro and weighted averages. Pay close attention to the metrics for class 1 (the catastrophic event).

<nav aria-label="Return navigation for code cell 63">
<a href="#read-code-63" aria-label="Go back and read code cell 63">&#8617; Go back and read the code cell</a>
</nav>

### Key Metric Formulas

| Metric | Formula | What It Measures |
| :--- | :--- | :--- |
| Precision | TP / (TP + FP) | Of all predicted catastrophes, how many were real? |
| Recall | TP / (TP + FN) | Of all real catastrophes, how many did the model find? |
| F1-Score | 2 × (P × R) / (P + R) | Harmonic mean — penalizes extreme imbalance between P and R |
| Accuracy | (TN + TP) / Total | Overall correctness — unreliable when classes are imbalanced |

### ✏️ Explanation — Precision and Recall

Answer the following in complete sentences:

1. What is the **precision** for class 1? In plain terms: out of every time your model predicted a catastrophe, what fraction was it correct?

2. What is the **recall** for class 1? Out of every actual catastrophe in the test set, what fraction did your model catch?

3. Is your F1-score closer to your precision or your recall? What does that imply about which type of error the model makes more often?

4. Given your story's stakes, should this model optimize for **precision** or **recall**? Justify your answer in 2–3 sentences.

---
*Write your response here.*

---

<a name="read-code-64"></a>

**Cell 64 — Bias-Variance Decomposition**

This cell uses `mlxtend` to decompose your model's total error into bias, variance, and noise. High bias = underfitting (model is too simple). High variance = overfitting (model memorized training noise). Install mlxtend first if needed.

<nav aria-label="Code cell 64 navigation">
<a href="#skip-code-64" aria-label="Skip code cell 64 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# pip install mlxtend -q

# from mlxtend.evaluate import bias_variance_decomp

# avg_expected_loss, avg_bias, avg_var = bias_variance_decomp(
#     model, X_train.values, y_train.values, X_test.values, y_test.values,
#     loss='0-1_loss', random_seed=42)

# print(f'Average expected loss: {avg_expected_loss:.3f}')
# print(f'Average bias:          {avg_bias:.3f}')
# print(f'Average variance:      {avg_var:.3f}')

<a name="skip-code-64"></a>

**Expected output:** Three decimal values for expected loss, bias, and variance. The expected loss approximately equals bias + variance + noise. Use these values in the explanation cell below.

<nav aria-label="Return navigation for code cell 64">
<a href="#read-code-64" aria-label="Go back and read code cell 64">&#8617; Go back and read the code cell</a>
</nav>

### ✏️ Explanation — Bias-Variance

Answer the following in complete sentences:

1. Report your model's bias, variance, and expected loss values.

2. Is **bias** or **variance** the larger contributor to your model's error? Is the model underfitting or overfitting?

3. Compare your training accuracy and test accuracy from Cell 61. Does the gap align with what the bias-variance decomposition shows? Explain why or why not.

4. Name one concrete change you could make to reduce the dominant source of error — for example: add more features, reduce features, adjust regularization strength, or collect more data.

---
*Write your response here.*

---

---
# Submission Checklist

Before sharing your link, confirm all of the following:

- Your name is at the top of the notebook (not your student ID)
- All 8 parts are complete and all cells have been run
- All written explanation cells have responses in complete sentences
- Each visualization has a descriptive Markdown cell below it explaining what it shows
- Your story mapping table is complete with all columns accounted for
- Your pipeline reflection (Part 6) is written in the Markdown cell
- The file is named `Your_Name.ipynb`

**To share:** Go to **Share** in Colab → set access to **Anyone with the link can view** → copy the link → paste into Canvas.